# CNT RPA analysis and presentation plots

This notebook collects the CNT plots used to validate and present the equilibrium RPA workflow:

1. Band structure centered at the calculated intrinsic midgap.
2. Momentum-resolved RPA polarization at representative finite q-points.
3. Equilibrium finite-device GW/NEGF polarization versus periodic q-averaged RPA polarization.
4. Effective RPA dielectric function and loss response.

The GW/NEGF and RPA comparison uses equal contact chemical potentials and one SCBA iteration. RPA uses periodic Bloch eigenstates; GW/NEGF uses the finite transport-device Green's functions.

In [ ]:
from pathlib import Path
import datetime as dt
import numpy as np
import matplotlib.pyplot as plt
import tomllib

plt.style.use("default")
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "stix",
    "figure.figsize": (7.1, 4.35),
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "axes.titleweight": "semibold",
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "legend.frameon": False,
    "legend.fontsize": 9,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.04,
    "pdf.fonttype": 42,
})

# Legacy settings retained below are overridden by the thesis style above.
plt.rcParams.update({
    "figure.figsize": (8.8, 5.0),
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.labelsize": 12,
    "axes.titlesize": 14,
    "legend.frameon": False,
    "legend.fontsize": 9,
    "savefig.dpi": 220,
})

def find_w90_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for parent in (start, *start.parents):
        if (parent / "data_analysis").is_dir() and (parent / "carbon-nanotube").is_dir():
            return parent
    raise RuntimeError("Could not find w90 root.")

def load(path, *, mmap=False):
    return np.load(path, mmap_mode="r" if mmap else None, allow_pickle=True)

def nearest_index(values, target):
    return int(np.argmin(np.abs(values - target)))

def export_figure(fig, filename):
    path = PRESENTATION_OUTPUTS / filename
    fig.savefig(path, bbox_inches="tight")
    print("Exported:", path)
    return path

W90_ROOT = find_w90_root()
DATA_ANALYSIS_ROOT = W90_ROOT / "data_analysis"
ROOT = W90_ROOT / "carbon-nanotube" / "gw-unit-cell"
VALIDATION_ROOT = DATA_ANALYSIS_ROOT / "validation_outputs" / "cnt_hbn_general" / "carbon_nanotube"
PRESENTATION_OUTPUTS = DATA_ANALYSIS_ROOT / "presentation_outputs" / "carbon_nanotube"
PRESENTATION_OUTPUTS.mkdir(parents=True, exist_ok=True)

RPA_CONFIG = ROOT / "quatrex_config_rpa_smooth.toml"
NEGF_CONFIG = ROOT / "quatrex_config_equilibrium_validation.toml"
RPA_RAW = ROOT / "outputs_RPA_smooth" / "raw_rpa_debug"
GW_OUT = ROOT / "outputs_equilibrium_validation"

with NEGF_CONFIG.open("rb") as file:
    negf_config = tomllib.load(file)

print("RPA config:", RPA_CONFIG)
print("GW/NEGF config:", NEGF_CONFIG)
print("Presentation outputs:", PRESENTATION_OUTPUTS)

## 1. Electronic structure: intrinsic CNT band structure

The energy zero is the midgap calculated directly from the CNT band eigenvalues. This is the appropriate presentation reference for an intrinsic, undoped CNT. It does not alter the Hamiltonian or eigenvalues.

In [ ]:
BAND_DATA = VALIDATION_ROOT / "band_structure" / "validation_cnt_bandstructure_midgap.npz"
band = load(BAND_DATA)
k_over_pi = band["k_points"] / np.pi
band_energies = band["plotted_eigenvalues"]

fig, ax = plt.subplots(figsize=(8.8, 5.2), constrained_layout=True)
for values in band_energies.T:
    ax.plot(k_over_pi, values, color="#1f4e79", linewidth=0.9)
ax.axhline(0.0, color="#b22222", linestyle="--", linewidth=1.1, label=r"$E_\mathrm{midgap}$")
ax.set(xlabel=r"$k/\pi$", ylabel=r"$E-E_\mathrm{midgap}$ (eV)",
       title="CNT Band Structure")
ax.set_xlim(k_over_pi[0], k_over_pi[-1])
ax.set_ylim(-3, 3)
ax.legend(loc="upper right")
ax.grid(True, alpha=0.25, linewidth=0.5)
export_figure(fig, "cnt_bandstructure_midgap.png")
plt.show()

## 2. Momentum-resolved equilibrium RPA polarization

Each line is the scalar RPA polarization response at one momentum transfer q. It is not a q-average. The q = 0 density-response line vanishes in this implementation because the uniform density operator cannot create transitions between orthogonal Bloch eigenstates.

In [ ]:
POLARIZATION_DATA = VALIDATION_ROOT / "polarization_behavior" / "validation_cnt_rpa_polarization.npz"
pol = load(POLARIZATION_DATA)
q_validation = pol["q_points"]
frequencies_validation_eV = pol["frequencies"]
p_validation = pol["p_base"]
q_fractions_to_plot = [0.25, 0.50]
for q_fraction in q_fractions_to_plot:
    q_index = nearest_index(q_validation, q_fraction * np.pi)
    q_label = r"$q = \pi/4$" if np.isclose(q_fraction, 0.25, atol=0.02) else r"$q = \pi/2$"
    q_filename = "q_pi_over_4" if np.isclose(q_fraction, 0.25, atol=0.02) else "q_pi_over_2"
    fig, ax = plt.subplots(figsize=(10.8, 5.0), constrained_layout=True)
    ax.plot(
        frequencies_validation_eV,
        p_validation[q_index].real,
        color="tab:blue",
        linewidth=1.8,
        label=r"Re $P(q,\omega)$",
    )
    ax.plot(
        frequencies_validation_eV,
        p_validation[q_index].imag,
        color="tab:orange",
        linewidth=1.8,
        label=r"Im $P(q,\omega)$",
    )
    ax.axhline(0.0, color="0.35", linewidth=0.7)
    ax.set(
        title=f"CNT RPA Polarization at {q_label}",
        xlabel="Energy transfer, $\\hbar\\omega$ (eV)",
        ylabel="Polarization response",
    )
    ax.legend()
    ax.grid(True, alpha=0.25, linewidth=0.5)
    export_figure(fig, f"cnt_rpa_polarization_{q_filename}.png")
    plt.show()

## 3. Equilibrium polarization: finite-device GW/NEGF versus periodic RPA

- Black: trace of the finite-device GW/NEGF retarded polarization, divided by the finite-device-to-unit-cell basis ratio.
- Blue: trace of each periodic unit-cell RPA polarization matrix, averaged over q.

The traces are comparable diagnostics, but they are not mathematically identical observables because one comes from a finite transport representation and the other from a periodic Bloch representation.

In [ ]:
required = [
    GW_OUT / "p_retarded_diagonal_0.npy",
    RPA_RAW / "frequencies_eV.npy",
    RPA_RAW / "q_points.npy",
    RPA_RAW / "polarization_retarded_qw.npy",
]
missing = [path for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing comparison outputs:\n" + "\n".join(map(str, missing)))

transport_energies = np.linspace(
    negf_config["electron"]["energy_window_min"],
    negf_config["electron"]["energy_window_max"],
    negf_config["electron"]["energy_window_num"],
)
response_energies_eV = transport_energies - transport_energies[0] + 1e-6

p_gw_diagonal = np.asarray(load(GW_OUT / "p_retarded_diagonal_0.npy", mmap=True))
p_gw_trace = p_gw_diagonal.sum(axis=-1)
rpa_freqs_eV = np.asarray(load(RPA_RAW / "frequencies_eV.npy"))
q_raw = np.asarray(load(RPA_RAW / "q_points.npy"))
p_rpa = load(RPA_RAW / "polarization_retarded_qw.npy", mmap=True)
p_rpa_trace_q_mean = np.asarray(np.trace(p_rpa, axis1=2, axis2=3)).mean(axis=0)

basis_ratio = p_gw_diagonal.shape[-1] / p_rpa.shape[-1]
p_gw_trace_per_unit_cell = p_gw_trace / basis_ratio

print(f"GW diagonal length: {p_gw_diagonal.shape[-1]}")
print(f"RPA unit-cell orbital count: {p_rpa.shape[-1]}")
print(f"Finite-device/unit-cell basis ratio: {basis_ratio:g}")
print("GW/NEGF output modified:", dt.datetime.fromtimestamp((GW_OUT / "p_retarded_diagonal_0.npy").stat().st_mtime))
print("RPA output modified:", dt.datetime.fromtimestamp((RPA_RAW / "polarization_retarded_qw.npy").stat().st_mtime))

fig, ax = plt.subplots(figsize=(8.8, 5.2), constrained_layout=True)
ax.plot(response_energies_eV, p_gw_trace_per_unit_cell.real, color="black", linewidth=1.9, label=f"Re GW/NEGF / {basis_ratio:g} cells")
ax.plot(response_energies_eV, p_gw_trace_per_unit_cell.imag, color="black", linestyle="--", linewidth=1.9, label=f"Im GW/NEGF / {basis_ratio:g} cells")
ax.plot(rpa_freqs_eV, p_rpa_trace_q_mean.real, color="tab:blue", linewidth=1.9, label="Re q-avg RPA")
ax.plot(rpa_freqs_eV, p_rpa_trace_q_mean.imag, color="tab:blue", linestyle="--", linewidth=1.9, label="Im q-avg RPA")
ax.axhline(0.0, color="0.35", linewidth=0.7)
ax.set(title="CNT Equilibrium Polarization: GW/NEGF and RPA",
       xlabel="Energy transfer, $\\hbar\\omega$ (eV)", ylabel="Polarization trace per unit cell")
ax.legend(ncol=2)
ax.grid(True, alpha=0.25, linewidth=0.5)
export_figure(fig, "cnt_equilibrium_polarization_negf_vs_rpa.png")
plt.show()

## 4. Absolute-magnitude validation: periodic RPA versus periodic Green-function bubble

This controlled comparison evaluates the polarization using two independent
formulations with the same periodic CNT Bloch Hamiltonian,
occupations, q-points, broadening, density vertex, and spin degeneracy:

- the implemented band-sum RPA;
- the equilibrium Green-function bubble underlying the GW polarization.

No normalization or fitted scaling is applied to either curve. Agreement therefore
validates the RPA complex response and its absolute magnitude.


In [ ]:
PERIODIC_VALIDATION = PRESENTATION_OUTPUTS / "cnt_periodic_rpa_vs_gf_bubble.npz"
if not PERIODIC_VALIDATION.exists():
    raise FileNotFoundError("Missing periodic RPA/GF validation data. Run data_analysis/scripts/validate_periodic_rpa_vs_gf_bubble.py cnt first.")

periodic_validation = load(PERIODIC_VALIDATION)
validation_frequency = periodic_validation["frequencies_eV"]
validation_q_over_pi = periodic_validation["q_over_pi"]
validation_rpa = periodic_validation["rpa_trace"]
validation_gf = periodic_validation["gf_trace"]

for index, q_fraction in enumerate(validation_q_over_pi):
    q_label = r"$q \approx \pi/4$" if np.isclose(q_fraction, 0.25, atol=0.02) else r"$q = \pi/2$"
    q_filename = "q_pi_over_4" if np.isclose(q_fraction, 0.25, atol=0.02) else "q_pi_over_2"
    fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.8), constrained_layout=True)

    for axis, component, component_label in zip(
        axes, ("real", "imag"), (r"$\mathrm{Re}\,P(q,\omega)$", r"$\mathrm{Im}\,P(q,\omega)$")
    ):
        rpa_component = getattr(validation_rpa[index], component)
        gf_component = getattr(validation_gf[index], component)
        component_error = np.linalg.norm(gf_component - rpa_component) / np.linalg.norm(rpa_component)

        axis.plot(validation_frequency, rpa_component, linewidth=2.1, label="Band-sum RPA")
        axis.plot(validation_frequency, gf_component, "--", linewidth=1.9, label="Periodic GF bubble")
        axis.axhline(0.0, color="0.35", linewidth=0.8)
        axis.set_title(f"{component_label}\nrelative error = {100 * component_error:.3f}%")
        axis.set_xlabel("Energy transfer (eV)")
        axis.set_ylabel("Polarization trace")
        axis.grid(True, alpha=0.25, linewidth=0.5)

    axes[0].legend()
    fig.suptitle(f"CNT Periodic Polarization Validation, {q_label}")
    export_figure(fig, f"cnt_periodic_rpa_vs_gf_{q_filename}.png")
    plt.show()


## 5. Effective RPA dielectric response

The effective dielectric response is a physically selected positive Coulomb-mode projection. The loss function is therefore more meaningful than an unprojected trace over the full dielectric matrix.

In [ ]:
DIELECTRIC_DATA = VALIDATION_ROOT / "dielectric_function" / "validation_cnt_rpa_effective_dielectric_dominant_positive.npz"
diel = load(DIELECTRIC_DATA)
q_dielectric = diel["q_points"]
frequencies_dielectric_eV = diel["frequencies"]
epsilon_eff = diel["epsilon_eff"]
loss_eff = diel["loss_eff"]
projection = str(diel["projection"]) if "projection" in diel.files else "effective positive Coulomb-mode projection"
q_indices = [nearest_index(q_dielectric, fraction * np.pi) for fraction in (0.25, 0.50)]
colors = ["tab:blue", "tab:orange"]

fig, axes = plt.subplots(1, 3, figsize=(13.8, 4.4), constrained_layout=True)
for q_index, color in zip(q_indices, colors):
    label = f"q/pi = {q_dielectric[q_index] / np.pi:.2f}"
    axes[0].plot(frequencies_dielectric_eV, epsilon_eff[q_index].real, label=label, linewidth=1.9, color=color)
    axes[1].plot(frequencies_dielectric_eV, epsilon_eff[q_index].imag, label=label, linewidth=1.9, color=color)
    axes[2].plot(frequencies_dielectric_eV, loss_eff[q_index], label=label, linewidth=1.9, color=color)
axes[0].axhline(1.0, color="0.45", linewidth=0.8, linestyle=":")
axes[1].axhline(0.0, color="0.45", linewidth=0.8)
axes[2].axhline(0.0, color="0.45", linewidth=0.8)
axes[0].set_title(r"Re $\epsilon_\mathrm{eff}(q,\omega)$")
axes[1].set_title(r"Im $\epsilon_\mathrm{eff}(q,\omega)$")
axes[2].set_title(r"Loss $-\mathrm{Im}[1/\epsilon_\mathrm{eff}]$")
for ax in axes:
    ax.set_xlabel("Energy transfer, $\\hbar\\omega$ (eV)")
    ax.legend()
    ax.grid(True, alpha=0.25, linewidth=0.5)
axes[0].set_ylabel("Effective dielectric response")
fig.suptitle("CNT RPA Effective Dielectric Response", fontsize=15)
export_figure(fig, "cnt_rpa_effective_dielectric.png")
plt.show()
print("Projection:", projection)

## Presentation summary

Use the band structure and momentum-resolved q-lines for physical verification. The
matched periodic RPA-versus-Green-function comparison is the primary implementation
and absolute-magnitude validation for CNT. Use the finite-device
GW/NEGF comparison as an integration and representation diagnostic, not as a strict
bulk-normalization test. The effective dielectric response presents the resulting
screening behavior. Exports are written to the material's presentation-output folder.


## 6. Individual q-point dielectric response: real and imaginary components

Separate plots for each q-value showing the real and imaginary parts of the effective dielectric function. These highlight how screening evolves with momentum transfer.

In [ ]:
# Generate individual q-point plots for CNT
q_fractions_to_plot = [0.25, 0.50]  # q/pi = 0.25 and q/pi = 0.50

for q_fraction in q_fractions_to_plot:
    q_index = nearest_index(q_dielectric, q_fraction * np.pi)
    q_actual = q_dielectric[q_index] / np.pi
    
    # Create descriptive label
    if np.isclose(q_fraction, 0.25, atol=0.02):
        q_label = r"$q = \pi/4$"
        q_filename = "q_pi_over_4"
    elif np.isclose(q_fraction, 0.50, atol=0.02):
        q_label = r"$q = \pi/2$"
        q_filename = "q_pi_over_2"
    else:
        q_label = f"$q \\approx {q_actual:.2f}\\pi$"
        q_filename = f"q_{q_actual:.2f}pi"
    
    # Create figure with real and imaginary parts on the same axis
    fig, ax = plt.subplots(figsize=(10.8, 5.0), constrained_layout=True)
    ax.plot(
        frequencies_dielectric_eV,
        epsilon_eff[q_index].real,
        linewidth=2.2,
        color="tab:blue",
        label=r"Re $\epsilon_\mathrm{eff}(q,\omega)$",
    )
    ax.plot(
        frequencies_dielectric_eV,
        epsilon_eff[q_index].imag,
        linewidth=2.2,
        color="tab:orange",
        label=r"Im $\epsilon_\mathrm{eff}(q,\omega)$",
    )
    ax.axhline(1.0, color="0.45", linewidth=0.8, linestyle=":", label="Vacuum")
    ax.axhline(0.0, color="0.35", linewidth=0.7)
    ax.set_title(f"CNT Effective Dielectric Response at {q_label}", fontsize=15)
    ax.set_xlabel("Energy transfer, $\\hbar\\omega$ (eV)")
    ax.set_ylabel("Effective dielectric response, $\\epsilon_\\mathrm{eff}$")
    ax.legend()
    ax.grid(True, alpha=0.25, linewidth=0.5)

    export_figure(fig, f"cnt_epsilon_eff_{q_filename}.png")
    plt.show()

## Matched-background dielectric response (`epsilon_r = 1`)

This diagnostic reconstructs the periodic dielectric response using the same background permittivity and dominant-positive Coulomb projection used in the CNT-versus-hBN comparison. It displays the result only; it does not export a PNG.

In [ ]:
matched_data = load(
    DATA_ANALYSIS_ROOT
    / "presentation_outputs"
    / "matched_background"
    / "cnt_hbn_matched_epsilon1_dielectric_comparison.npz"
)
matched_q = matched_data["cnt_q_points"]
matched_frequencies = matched_data["cnt_frequencies_eV"]
matched_epsilon = matched_data["cnt_epsilon_eff"]

q_fractions_to_plot = [0.25, 0.50]
energy_mask = matched_frequencies <= 8.0

for q_fraction in q_fractions_to_plot:
    q_index = nearest_index(matched_q / np.pi, q_fraction)
    q_actual = matched_q[q_index] / np.pi

    if np.isclose(q_fraction, 0.25, atol=0.02):
        q_label = r"$q = \pi/4$"
    elif np.isclose(q_fraction, 0.50, atol=0.02):
        q_label = r"$q = \pi/2$"
    else:
        q_label = f"$q \\approx {q_actual:.2f}\\pi$"

    fig, ax = plt.subplots(figsize=(10.8, 5.0), constrained_layout=True)
    ax.plot(
        matched_frequencies[energy_mask],
        matched_epsilon[q_index, energy_mask].real,
        linewidth=2.2,
        color="tab:blue",
        label=r"Re $\epsilon_\mathrm{eff}(q,\omega)$",
    )
    ax.plot(
        matched_frequencies[energy_mask],
        matched_epsilon[q_index, energy_mask].imag,
        linewidth=2.2,
        color="tab:orange",
        label=r"Im $\epsilon_\mathrm{eff}(q,\omega)$",
    )
    ax.axhline(1.0, color="0.45", linewidth=0.8, linestyle=":", label="Vacuum")
    ax.axhline(0.0, color="0.35", linewidth=0.7)
    ax.set_title(
        rf"CNT Effective Dielectric Response at {q_label} ($\epsilon_r = 1$)",
        fontsize=15,
    )
    ax.set_xlabel("Energy transfer, $\\hbar\\omega$ (eV)")
    ax.set_ylabel("Effective dielectric response, $\\epsilon_\\mathrm{eff}$")
    ax.legend()
    ax.grid(True, alpha=0.25, linewidth=0.5)
    plt.show()
